


**Author:** Ethan Ligon



In this document we&rsquo;ll work through a simple approach to estimating
Constant Frisch Elasticity (CFE) demand systems.

There are two prerequisites; the first is the `CFEDemands` python
module.  You may have this already, but if not it  can be installed with a simple:



In [2]:
%pip install -r requirements-estimation.txt

Note: you may need to restart the kernel to use updated packages.


The second prerequisite is a dataset including detailed information on
household expenditures and characteristics.  The structure of that
dataset is detailed in the next section.



## Data



Consider the case in which the data are saved in a google sheet; we&rsquo;ll
turn data from these into `pandas` dataframes.  
There are limits on the allowable size of google sheets (5 million
cells when I last checked), so if your dataset is larger than this you&rsquo;ll need some other
mechanism to get your data into such dataframes.  Even in this latter
case you&rsquo;ll want your dataframes to have a structure that matches the
description given below.



### Structure of the Spreadsheet



An example spreadsheet with data for Uganda can be found at
[https://docs.google.com/spreadsheets/d/1yVLriVpo7KGUXvR3hq_n53XpXlD5NmLaH1oOMZyV0gQ/](https://docs.google.com/spreadsheets/d/1yVLriVpo7KGUXvR3hq_n53XpXlD5NmLaH1oOMZyV0gQ/)

This spreadsheet consists of three worksheets, labeled &ldquo;RDI&rdquo;, &ldquo;FCT&rdquo;,
&ldquo;Household Characteristics&rdquo;, &ldquo;Prices&rdquo;, and a series of different years of food &ldquo;Expenditures&rdquo;.

Each worksheet has the initial three columns labeled &ldquo;i&rdquo;, &ldquo;t&rdquo;, and
&ldquo;m&rdquo;; together these identify an observation uniquely.  The column &ldquo;i&rdquo;
should include a household id; the column &ldquo;t&rdquo; should include an
indicator for the year or period (for those of you with
cross-sectional data these will all take a common value, e.g., 2019);
the column &ldquo;m&rdquo; should be an indicator of &ldquo;market&rdquo;, which may be a
region identifier, or may take a single value (e.g., &ldquo;Malawi&rdquo;).

The triple (i,t,m) will serve as the index of some constructed
dataframes.  It is important not to have **too** many different values
for &ldquo;m&rdquo; (unless you have lots of data and RAM to play with).

Beyond the first three columns, the &ldquo;Expenditures&rdquo; worksheet for a given year should
have one column for each category of expenditures (e.g., &ldquo;Beans&rdquo;, &ldquo;Beef&rdquo;, &ldquo;Beer&rdquo;,&hellip;).

Beyond the first three columns, the &ldquo;Household Characteristics&rdquo; should
have columns corresponding to different elements of the vector of
observable characteristics $z$.  If the data allow, I recommend
constructing observables that match what I have in these data:
namely numbers of different children by age and sex, along with the
logarithm of total household size (&ldquo;log HSize&rdquo;).



### From Sheet to DataFrame to Result Object



We begin by defining a dictionary that contains the spreadsheet key. 
One is provided for the Ugandan example that I wish to work
through.



In [3]:
Uganda_Data = '1yFWlP5N7Aowaj6t2roRSFFUC50aFD-RLBGfzGtqLl0w'

With the spreadsheet defined , grab it and define a couple of
dataframes. (Sheets should be shared with `students@eep153.iam.gserviceaccount.com`):



In [30]:
import pandas as pd
import numpy as np
from ligonlibrary.sheets import read_sheets
NG_URL= "https://docs.google.com/spreadsheets/d/1rbWPdn_Rt3MRoFb_wkomHBcNVmM-3F3lNwCpxP5W_3g/edit?gid=1354121303#gid=1354121303" 
x = read_sheets(NG_URL,sheet='Food Expenditures (2016Q1)')

# Change 'Uganda' to key of your own sheet in Sheets, above
d = read_sheets(NG_URL,sheet="Household Characteristics")


d.columns.name = 'k'
# remove duplicate column names, keep first occurrence
d = d.loc[:, ~d.columns.duplicated()]

# Data comes in long format; pivot to wide (one column per food)
x = x.pivot_table(index=['i','t','m'],columns='j',values='Expenditure',
                   aggfunc='sum')
x = x.replace(0,np.nan) # Replace zeros with missing

# Take logs of expenditures; call this y
y = np.log(x)

d.set_index(['i','t','m'],inplace=True)

# NaN in demographic columns means zero people in that category
d = d.fillna(0)

# 1. Handle Household Characteristics (d)
# Ensure columns are unique before stacking
d = d.loc[:, ~d.columns.duplicated()]


This gives a dataframe of household characteristics $d$ with a simple structure.



In [31]:
d.head()

,,k,Female 00-03,Female 04-08,Female 09-13,Female 14-18,Female 19-30,Female 31-50,Female 51+,Female NA,Male 00-03,Male 04-08,Male 09-13,Male 14-18,Male 19-30,Male 31-50,Male 51+,Male NA,log HSize
i,t,m,,,,,,,,,,,,,,,,,
70001,2010Q3,North Central,1,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,1.386294
70002,2010Q3,North Central,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0.693147
70003,2010Q3,North Central,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0.000000
70004,2010Q3,North Central,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0.000000
70005,2010Q3,North Central,0,1,0,0,1,0,0,0,1,0,1,0,0,0,1,0,1.609438


Also a dataframe of log expenditures $y$ with a similarly simple
structure.  Note that any expenditures of zero are dropped and
replaced with a missing value indicator.



In [26]:
y.head()

,,j,Apples,Avocado pear,Baby milk powder,Bananas,Beef,Beer,Biscuits,Bottled water,Bread,Brown beans,...,Sweet potatoes,Tea,Tomato puree (canned),Tomatoes,Watermelon,Wheat flour,White beans,Wild game meat,Yam flour,Yam--roots
i,t,m,,,,,,,,,,,,,,,,,,,,,
10001,2016Q1,South east,NaN,NaN,NaN,4.210878,7.600902,NaN,4.248495,NaN,5.991465,NaN,...,5.08136,NaN,NaN,5.005598,NaN,NaN,5.521461,NaN,NaN,6.164674
10002,2016Q1,South east,4.787492,NaN,NaN,4.835033,6.907755,NaN,5.472671,5.703782,6.396930,5.521461,...,NaN,NaN,NaN,5.703782,5.422525,NaN,NaN,NaN,NaN,6.164674
10003,2016Q1,South east,NaN,NaN,NaN,4.605170,6.907755,NaN,3.688879,5.010635,5.603241,5.521461,...,NaN,5.192957,5.298317,5.181162,NaN,NaN,NaN,NaN,NaN,6.159585
10004,2016Q1,South east,NaN,NaN,NaN,NaN,6.907755,NaN,4.248495,NaN,5.577266,5.521461,...,NaN,5.010635,5.010635,5.181162,NaN,NaN,NaN,NaN,NaN,5.471527
10005,2016Q1,South east,NaN,NaN,NaN,NaN,6.907755,NaN,4.605170,NaN,5.991465,5.703782,...,NaN,5.010635,5.010635,5.586627,NaN,NaN,NaN,NaN,NaN,6.159585


## Estimation



Let $y_{i}^j$ be log expenditures on food $j$ by household $i$ at a particular time.  We want to estimate a regression that takes the form
$$
      y^j_{i} = A^j(p) + \gamma_j'd_i + \beta_j w_i + \zeta^j_i.
$$
So, a first step is to turn our dataframe for `y` into a series, so we can put it on the left-hand side of this regression.  Then we need to make our dataframe $d$ conform:



In [32]:
# Now stack it. This creates a Series with index [i, t, m, k]
d = d.stack().dropna()

# 2. Handle Food Expenditures (y)
# y is already pivoted, so stacking it creates index [i, t, m, j]
y = y.stack().dropna()

# Check that indices are in right places!
assert y.index.names == ['i','t','m','j']
assert d.index.names == ['i','t','m','k']

### Basic Estimation



Just one line to set up the regression:



In [33]:
from cfe import Regression

result = Regression(y=y,d=d)

ValueError: Index contains duplicate entries, cannot reshape

And then one line to predict expenditures and estimate most of the things we need:



In [ ]:
result.predicted_expenditures()

Now we can compare predicted log expenditures with actual:



In [ ]:
%matplotlib widget
df = pd.DataFrame({'y':y,'yhat':result.get_predicted_log_expenditures()})
df.plot.scatter(x='yhat',y='y')

That&rsquo;s all there is to estimation!  Note that we didn&rsquo;t estimate
demands for all goods&mdash;lots of goods didn&rsquo;t have enough observations,
and were automatically dropped.



### Parameters



#### (Relative) Income Elasticity



Some of the parameters have an interesting interpretation.  First among these are the $\beta$ coefficients, which govern how *income elastic* different goods are (higher values means more elastic).  These are also called *Frisch elasticities*.



In [ ]:
result.get_beta().sort_values()

Here we use a plot to visualize, with confidence intervals.



In [ ]:
result.graph_beta()

#### Demand and Household Composition



We&rsquo;re particularly interested in the effects of household composition on demand.  These are captured by the $\gamma_j$ parameters in the regression.



In [ ]:
result.gamma

### Saving Result



You can save the estimated result, using something like



In [ ]:
result.to_pickle('my_estimates.pickle')

You can subsequently load this using



In [ ]:
import cfe
result = cfe.regression.read_pickle('my_estimates.pickle')